# Classification: late-order risk at carrier handover

**Business role.** Rank handed-over orders by their risk of missing Olist's existing estimated delivery date, so operations can review a limited queue. The target is derived from actual delivery and estimated delivery dates; no post-delivery outcome is a predictor. The regression's time estimate supports the review, while this classifier supplies the ranking.

The training, fold construction and preprocessing code live in [`phase2_handover_classifier.py`](../../phase2_handover_classifier.py) and imported helpers. This notebook reads saved outputs; it does not refit or use the already examined later period for model selection. Run `python phase2_handover_classifier.py` from `phase2_final/` for a full rerun.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data" / "phase2_feature_spec.json").is_file()
             and (p / "phase2_eta.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from phase2_final/ or a folder beneath it")
RESULTS = ROOT / "results" / "phase2"
spec = json.loads((ROOT / "data" / "phase2_feature_spec.json").read_text())
print("Phase 2 folder:", ROOT)
print("Saved feature contract:", spec.get("version", "version unspecified"))

Phase 2 folder: /Users/joshualum/Documents/VSCode/IT5006_Grp_6/phase2_final
Saved feature contract: version unspecified


## Validation comparison and model choice

PR-AUC is the main ranking measure because late orders are uncommon. The saved main search compares regularised logistic variants, a single tree, random forests and gradient boosting. A separate audit fits unregularised logistic regression on the same five windows, so its plain-family baseline is shown alongside the saved search. The selected model comes from chronological validation; model selection does not use the later-period rows below.

In [2]:
validation = pd.read_csv(RESULTS / "handover" / "validation_summary.csv")
plain_windows = pd.read_csv(RESULTS / "handover" / "unregularized_baseline_audit" / "validation_windows.csv")
plain = (plain_windows.loc[plain_windows.variant.eq("primary")]
         .groupby("candidate", as_index=False)[["pr_auc", "roc_auc", "recall_top10", "precision_top10"]].mean())
selection = json.loads((RESULTS / "handover" / "selection.json").read_text())
primary = selection["variants"]["primary"]
print("Selected:", primary["selected"])
print("Reason:", primary["reason"])
key = ["logistic_c1", primary["selected"], "tree_plain", primary["tuned"]["tree"],
       primary["tuned"]["forest"], primary["tuned"]["boost"]]
comparison = pd.concat([plain, validation.loc[validation.variant.eq("primary") & validation.candidate.isin(key),
                                         plain.columns]], ignore_index=True)
display(comparison.sort_values("pr_auc", ascending=False).round(3).reset_index(drop=True))

Selected: logistic_c0.01
Reason: best forest_l50_sqrt (mean validation PR-AUC 0.3686); chosen logistic_c0.01 (0.3666), the simplest line-up entry within one standard error of the best


,candidate,pr_auc,roc_auc,recall_top10,precision_top10
0,forest_l50_sqrt,0.369,0.747,0.343,0.332
1,logistic_c0.01,0.367,0.769,0.363,0.348
2,boost_lr0.03_n200_l20,0.354,0.741,0.333,0.316
3,logistic_c1,0.343,0.739,0.349,0.332
4,logistic_unregularized,0.337,0.733,0.344,0.327
5,tree_d8_l100,0.308,0.684,0.322,0.303
6,tree_plain,0.138,0.561,0.213,0.201


## Later-period ranking and review workload

The 14% review share was chosen using validation under an illustrative five-to-one value ratio. It is an evaluation setting, not a measured staff capacity. The later period had fewer late orders, so precision and the illustrative net score should be read in that context. The full saved comparison includes diagnostic rankings, while the report's operational proposal is a capacity-limited pilot.

In [3]:
test = pd.read_csv(RESULTS / "handover" / "test_metrics.csv")
display(test.loc[test.variant.eq("primary"),
                 ["ranking", "k", "precision", "recall", "pr_auc", "late_rate", "net_benefit_per_order"]]
        .round(3).reset_index(drop=True))
importance = pd.read_csv(RESULTS / "handover" / "primary_permutation_importance.csv")
display(importance.head(10).round(3))

,ranking,k,precision,recall,pr_auc,late_rate,net_benefit_per_order
0,model,0.14,0.118,0.473,0.231,0.035,-0.057
1,slack_rule,0.14,0.162,0.647,0.519,0.035,-0.027
2,regression_gap,0.14,0.168,0.673,0.500,0.035,-0.023
3,random,0.14,0.035,0.140,0.035,0.035,-0.116


,feature,mean_pr_auc_drop
0,handover_days,0.130
1,remaining_slack,0.085
2,promised_days,0.027
3,route_typical_days,0.019
4,distance_km,0.015
5,customer_state,0.012
6,same_state,0.003
7,n_items,0.003
8,total_freight,0.002
9,seller_state,0.001


## Rebuild the evidence

Run `python phase2_handover_classifier.py` from `phase2_final/` to refit candidates and regenerate `results/phase2/handover/`; then rerun this notebook. The script fits preprocessing within the training folds and uses outcomes available before each validation window. The [report evidence script](../../reports/phase2_final_evidence.py) updates generated report tables and figures.